# Create a mock galaxy to be fit with Bagpipes and Prospector

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np 
import matplotlib as mpl
mpl.rcParams["font.family"] = "serif"  # override bagpipes' Helvetica request
mpl.rcParams["text.usetex"] = True

from astropy.cosmology import WMAP9 as cosmo
import bagpipes as pipes
import matplotlib.pyplot as plt
#%matplotlib inline

from astropy.io import fits
from astropy.table import Table
from sedpy import observate

import fitutils as fit

In case of bugs with matplotlib

In [ ]:
import matplotlib as mpl
import os
import shutil

os.environ["PATH"] += os.pathsep + "/Library/TeX/texbin"
# This will find and delete the font cache folder
cache_dir = mpl.get_cachedir()
if os.path.exists(cache_dir):
#    shutil.rmtree(cache_dir)
    print("Cache cleared! Restart your Python kernel/IDE.")

latex_path = shutil.which("latex")
pdflatex_path = shutil.which("pdflatex")

print(f"LaTeX path: {latex_path}")
print(f"PDFLaTeX path: {pdflatex_path}")

if not latex_path:
    print("❌ LaTeX was not found in your system's PATH.")
else:
    print("✅ LaTeX is installed!")

## Generate filter transmission curves for missing JWST/MIRI bands

In [ ]:
print(observate.list_available_filters())

MIRI = ['jwst_f560w', 'jwst_f1130w', 'jwst_f1280w', 'jwst_f1500w', 'jwst_f2550w']
data = observate.load_filters(MIRI)
for filt in data:
    name = filt.name.split('_')[-1]
    wave = filt.wavelength
    trans = filt.transmission
    
    # Combine wavelength and transmission into a 2D array (column-wise)
    output_data = np.column_stack((wave, trans))
    
    # Define a filename based on the filter name
    
    filename = f"/Users/benjamincollins/University/PhD/Code/bagpipes/BlueJay/filters/{name}"
    
    # Save to file with a helpful header
    np.savetxt(
        filename, 
        output_data, 
        #fmt=['%.4f', '%.6f'], 
        comments=''
    )
    
    print(f"Saved {filename}")

## Generate boxcar filter transmission curves for ALMA bands 6 and 7

In [ ]:
# Band 6
fit.write_alma_transmission_curves("alma_band6", 
                               central_freq_ghz=233, 
                               bandwidth_ghz=7.5)


fit.write_alma_transmission_curves("alma_band7", 
                               central_freq_ghz=343.5, 
                               bandwidth_ghz=7.5)

# Set galaxy parameters

In [ ]:
mock_id = 9997

fit_instructions = {}                       # The fit instructions dictionary

# Nebular component
nebular = {}
nebular["logU"] = -3.0                # Log_10 of the ionisation parameter.
"""
exp = {}                          # Tau model star formation history component
exp["age"] = 2.2                 # Gyr
exp["tau"] = 0.75                 # Gyr
exp["massformed"] = 10.5            # log_10(M*/M_solar)
exp["metallicity"] = 0.1          # Z/Z_oldsolar

lognormal = {}
lognormal["massformed"] = 10.5            # log_10(M*/M_solar)
lognormal["metallicity"] = 0.1          # Z/Z_oldsolar
lognormal["tmax"] = 2.0
lognormal["fwhm"] = 0.5
"""

dblplaw = {}
dblplaw["massformed"] = 10.5
dblplaw["metallicity"] = 0.1
dblplaw["tau"] = 2.5
dblplaw["alpha"] = 18.0
dblplaw["beta"] = 2.5

zred = 2.00 # Set custom redshift

# Dust absorption parameters
dust = {}                               # Dust component
dust["type"] = "CF00"                   # Define the shape of the attenuation curve
dust["Av"] = 1.1                  # Vary Av between 0 and 3.2 magnitudes
dust["n"] = 0.5                 # Vary the slope of the attenuation curve from -1.0 to 1.5

# Dust emission parameters (now free parameters)
dust["qpah"] = 2.0                  # PAH mass fraction (spanning the entire grid)
dust["umin"] = 5.0                   # Lower limit of starlight intensity distribution (spanning the entire grid)
dust["gamma"] = 0.1                   # Fraction of stars at Umin

model_components = {}                   # The model components dictionary
model_components["nebular"] = nebular
model_components["redshift"] = zred      # Observed redshift  
#model_components["exponential"] = exp   
#model_components["lognormal"] = lognormal
model_components["dblplaw"] = dblplaw
model_components["dust"] = dust

print(model_components)

filt_list = np.loadtxt("filters/full_miri+alma67_filt_list.txt", dtype="str")   # Now using all available MIRI bands
model = pipes.model_galaxy(model_components, filt_list=filt_list, phot_units="mujy")

fig = model.plot()
fig = model.sfh.plot()
model.plot_full_spectrum()

# Get model photometry

In [ ]:
true_flux = model.photometry # This is your 'noiseless' truth

flux_err = true_flux * 0.1  # Add 10% flux error and perturb the observations
mock_flux = np.random.normal(loc=true_flux, scale=flux_err)

# Quick diagnostic printout
for i in range(len(true_flux)):
    print(f"Filter {i:02d} | True: {true_flux[i]:.2e} | Mock Observed: {mock_flux[i]:.2e} +/- {flux_err[i]:.2e}") 

# Manually extracted the labels from the results
labels = ['logmass', 'logzsol', 'dust2', 'duste_gamma', 'dust_index', 'duste_qpah', 'duste_umin', 'gas_logu']   

true_params = {}
true_params['redshift'] = model_components["redshift"]  # Now also storing redshift
true_params['logmass'] = model_components['dblplaw']['massformed']
true_params['logzsol'] = np.log10(model_components['dblplaw']['metallicity'])
true_params['dust2'] = model_components['dust']['Av']
true_params['duste_gamma'] = model_components['dust']['gamma']
true_params['dust_index'] = model_components['dust']['n']
true_params['duste_qpah'] = model_components['dust']['qpah']
true_params['duste_umin'] = model_components['dust']['umin']
true_params['gas_logu'] = model_components['nebular']['logU']

print(true_params)

# Create a dictionary of arrays to save
# Storing the raw fluxes and errors (assuming microjanskys)
fname = f"comparison/mock_fit/{mock_id}_mock.npz"
np.savez(
    fname, 
    flux=mock_flux, 
    flux_err=flux_err, 
    true_flux=true_flux,
    true_params=true_params
)
print(f"Mock data successfully written to {fname}")

photometry = np.c_[mock_flux, flux_err]

def load_mock_data(objid):
    print(f"Loading data for mock object {objid}")
    return photometry